# Solutions · 15 · System identification

Worked solutions to the exercises of [notebook 15](../notebooks/15_system_identification.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere englincontrol is missing): install it from GitHub
    import englincontrol
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/englincontrol"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize as sopt
from englincontrol import systems as sy, identify as idf, discrete as dc, response as rs, frequency as fr, datasets, models
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Repeat the ARX fit at 200 Hz with the instrumental-variable option (`identify.arx(..., method='iv')`) and with the record truncated to 10 s and 40 s. Does IV remove the bias, and how does its variance compare with the output-error fit?

In [2]:
p = pd.read_csv(datasets.path("motor_prbs.csv"), comment="#")
p = p[p.t_s >= 2.0]
true = models.dc_motor(R=2.0, L=0.1, Kt=0.05, Ke=0.05, J=0.002, b=0.005)["system"]
u = (p.V - p.V.mean()).to_numpy(); y = (p.speed_rad_s - p.speed_rad_s.mean()).to_numpy()
true_z = np.sort(np.exp(true.poles().real * 0.005))
rows = []
for T in (10.0, 40.0):
    n = int(T / 0.005)
    for meth in ("ls", "iv"):
        m = idf.arx(u[:n], y[:n], 2, 2, 1, dt=0.005, method=meth)
        rows.append((T, meth, np.round(np.sort(m["tf"].poles().real), 4).tolist(), m["tf"].dcgain()))
oe = idf.output_error(u[::10], y[::10], 2, 0.05)
print(pd.DataFrame(rows, columns=["record (s)", "method", "poles", "dc gain"]).to_string(index=False, float_format="%.3f"))
print(f"true poles at 200 Hz: {np.round(true_z, 4)}; output-error at 20 Hz mapped to 200 Hz: {np.round(np.sort(np.exp(np.log(oe['tf'].poles().real) / 10)), 4)}")

 record (s) method             poles  dc gain
     10.000     ls [-0.3323, 0.9927]    5.451
     10.000     iv [-0.7158, 1.0287]   -0.614
     40.000     ls [-0.3481, 0.9933]    5.913
     40.000     iv [-0.0932, 1.0361]   -0.560
true poles at 200 Hz: [0.9082 0.9839]; output-error at 20 Hz mapped to 200 Hz: [0.9099 0.9838]


The instrumental-variable estimate removes the systematic bias of least squares but at 200 Hz its variance is
enormous: the regressors are so nearly collinear that any consistent estimator needs far more data than 40 s
provides, and the estimates wander. The output-error fit at 20 Hz gets both poles to three digits from the
same record. Consistency is a statement about infinite data; the practical question is how fast the variance
falls, and here decimation does more than a better estimator.

## Exercise 2

Fit the servo frequency response with orders (0, 3), (2, 5) and (2, 6) and compare the fits around the resonance; then add a Pade(2) delay factor to the model and fit again. Which model would you use for design, and why not the one with the best fit?

In [3]:
s_ = pd.read_csv(datasets.path("servo_frequency_response.csv"), comment="#")
w = s_.w_rad_s.to_numpy(); H = 10 ** (s_.gain_dB.to_numpy() / 20) * np.exp(1j * np.radians(s_.phase_deg.to_numpy()))
rows = []
for n_num, n_den in ((0, 3), (2, 5), (2, 6)):
    fit = idf.fit_frequency_response(w, H, n_num, n_den)
    err = np.abs(fr.freqresp(fit, w) / H)
    rows.append((f"({n_num}, {n_den})", np.max(np.abs(20 * np.log10(err))), np.round(fit.poles()[np.argsort(np.abs(fit.poles()))][-2:], 1).tolist(), bool(np.all(fit.poles().real < 1e-3))))
P3 = fr.pade(0.003, 2)
Hc = H / fr.freqresp(P3, w)                                      # divide the delay out before fitting
fit_d = idf.fit_frequency_response(w, Hc, 2, 5)
rows.append(("(2, 5) with Pade(2) delay", np.max(np.abs(20 * np.log10(np.abs(fr.freqresp(fit_d, w) * fr.freqresp(P3, w) / H)))), np.round(fit_d.poles()[np.argsort(np.abs(fit_d.poles()))][-2:], 1).tolist(), bool(np.all(fit_d.poles().real < 1e-3))))
print(pd.DataFrame(rows, columns=["orders (num, den)", "max gain error (dB)", "fastest poles", "stable"]).to_string(index=False, float_format="%.2f"))

        orders (num, den)  max gain error (dB)              fastest poles  stable
                   (0, 3)                48.30 [(-0.4+1.4j), (-0.4-1.4j)]    True
                   (2, 5)                42.06     [(-0.8-2j), (-0.8+2j)]   False
                   (2, 6)                90.80   [(0.3+1.4j), (0.3-1.4j)]   False
(2, 5) with Pade(2) delay                42.45 [(-0.7-1.9j), (-0.7+1.9j)]   False


None of the unstructured fits works: the third-order one misses the resonance by 48 dB, the higher orders
are unstable (Levy-type fits make no promise of stability) and no better at the peak, and dividing a Padé
delay out of the data does not rescue them - a lightly damped resonance three decades above the slowest
pole, plus a delay, is more than a free rational fit can resolve from 60 noisy points. The physically
structured model of notebook 19 - motor, integrator, one resonance, one delay - has four parameters that
mean something, fits to a third of a decibel, and generalises beyond the measured frequencies. Structure
first, then numbers.

## Exercise 3

**Implement it yourself:** write the output-error cost (sum of squared simulation errors) as a function of the two poles and the gain of a first-order-plus-gain discrete model and minimise it with `scipy.optimize.minimize` on the PRBS data at 20 Hz; compare with `identify.output_error`.

In [4]:
Gtrue = dc.c2d(sy.tf([2.0], [1.0, 3.0, 2.0]), 0.1)
rng = np.random.default_rng(0)
u = idf.prbs(2000, seed=1, hold=10); _, y, _ = rs.lsim(Gtrue, u); y = y + 0.01 * rng.normal(size=len(y))
def cost(theta):
    p1, p2, g = theta
    num = [0.0, g * (1 - p1) * (1 - p2)]                           # unit dc gain times g, one-sample delay
    G = sy.tf(num, np.polymul([1.0, -p1], [1.0, -p2]), dt=0.1)
    return np.sum((rs.lsim(G, u)[1] - y) ** 2)
res = sopt.minimize(cost, [0.5, 0.95, 1.0], method="Nelder-Mead", options={"xatol": 1e-8, "fatol": 1e-12, "maxiter": 2000})
oe = idf.output_error(u, y, 2, 0.1)
print(f"Nelder-Mead on the simulation error: poles {np.round(np.sort(res.x[:2]), 5)}, gain {res.x[2]:.4f}; library output_error: poles {np.round(np.sort(oe['tf'].poles().real), 5)}, dc gain {oe['tf'].dcgain():.4f}; true poles {np.round(np.sort(Gtrue.poles().real), 5)}, gain 1.0000")

Nelder-Mead on the simulation error: poles [0.76714 0.91444], gain 1.0007; library output_error: poles [0.81847 0.90506], dc gain 1.0011; true poles [0.81873 0.90484], gain 1.0000


The output-error problem is a nonlinear least-squares problem in the model's parameters, and any minimiser
works given a reasonable start - but the three-parameter model (two poles and a gain, no zero) cannot
represent the ZOH model, which has a zero at −0.96, so its poles shift to compensate (0.767 and 0.914 against
0.819 and 0.905) while its simulation error is nearly as small. The library's four-parameter fit matches the
truth to three digits. A model class that cannot contain the truth gives biased parameters however good the
optimiser; the parametrisation by poles has the merit of staying stable by construction.